In [ ]:
import random
import time
from collections import deque
import threading
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- CSS Styling ---
custom_css = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght=400;500;600;700&family=JetBrains+Mono&display=swap');

.app-container { background-color: #f9f9ff; font-family: 'Inter', sans-serif; }
.modern-card { background-color: #ffffff; border: 1px solid #c3c6d7; border-radius: 12px; padding: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); box-sizing: border-box; }
.card-header { font-size: 18px; font-weight: 600; color: #141b2b; border-bottom: 1px solid #c3c6d7; padding-bottom: 10px; margin-bottom: 15px; display: flex; justify-content: space-between; }

.stat-box { background-color: #ffffff !important; border: 1px solid #c3c6d7 !important; border-radius: 12px !important; padding: 15px !important; box-sizing: border-box; }
.label-bold { font-weight: 600 !important; font-family: 'Inter', sans-serif !important; color: #141b2b !important; }

.puzzle-input input[type="number"] {
    font-size: 24px !important; font-weight: 700 !important; text-align: center !important; color: #004ac6 !important;
    background-color: #e1e8fd !important; border: 1px solid rgba(0,74,198,0.2) !important; border-radius: 8px !important;
    height: 100% !important; box-sizing: border-box;
}

.btn-primary { 
    background-color: #004ac6 !important; 
    color: white !important; 
    border-radius: 999px !important; 
    font-weight: 600 !important; 
    border: 1px solid #004ac6 !important; 
    width: 95% !important; 
    box-sizing: border-box !important;
}
.btn-primary:hover { background-color: #003ea8 !important; }

.btn-secondary { 
    background-color: #ffffff !important; 
    color: #141b2b !important; 
    border: 1px solid #c3c6d7 !important; 
    border-radius: 999px !important; 
    font-weight: 600 !important; 
    width: 95% !important; 
    box-sizing: border-box !important;
}
.btn-secondary:hover { background-color: #f1f3ff !important; }

.btn-action { background-color: #e1e8fd !important; color: #38485d !important; border-radius: 8px !important; font-weight: 600 !important; border: 1px solid rgba(0,74,198,0.2) !important; font-size: 12px !important; }
.btn-action:hover { opacity: 0.9 !important; }

.log-output { background-color: #f1f3ff !important; font-family: 'JetBrains Mono', monospace !important; border: none !important; }

.anim-board { display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; max-width: 320px; margin: 0 auto; background-color: #f1f3ff; padding: 20px; border-radius: 16px; position: relative; box-shadow: inset 0 2px 10px rgba(0,0,0,0.02); }
.anim-tile { aspect-ratio: 1; background-color: #ffffff; border: 1px solid #c3c6d7; border-radius: 12px; display: flex; align-items: center; justify-content: center; font-size: 32px; font-weight: 700; color: #004ac6; box-shadow: 0 2px 5px rgba(0,0,0,0.05); transition: all 0.3s ease; }
.anim-tile-empty { aspect-ratio: 1; background-color: rgba(220, 226, 247, 0.4); border: 2px dashed #c3c6d7; border-radius: 12px; }
</style>
"""

# --- UI Components ---
display(HTML(custom_css))

# Header Thanh tiêu đề chính
header_html = widgets.HTML(value="""
<div style="display: flex; justify-content: space-between; align-items: center; padding: 15px 30px; background-color: #ffffff; border-bottom: 1px solid #c3c6d7; font-family: 'Inter', sans-serif;">
    <span style="font-size: 20px; font-weight: 700; color: #141b2b;">8-Puzzle Solver Simulator</span>
    <div style="color: #004ac6; font-weight: 700; border-bottom: 2px solid #004ac6; padding-bottom: 4px; font-size: 14px;">AND-OR SEARCH</div>
</div>
""")

# 1. Khu vực Initial State (Bên trái)
input_boxes = [widgets.BoundedIntText(value=v, min=0, max=8, layout=widgets.Layout(width='auto', height='60px')) 
               for v in [1, 2, 3, 4, 0, 5, 7, 8, 6]]
for box in input_boxes: box.add_class('puzzle-input')

input_grid = widgets.GridBox(input_boxes, layout=widgets.Layout(grid_template_columns="repeat(3, 1fr)", gap="10px", margin="0 0 20px 0"))

btn_random = widgets.Button(description="Random", layout=widgets.Layout(flex='1'))
btn_random.add_class('btn-action')
btn_random.style.button_color = '#505f76'
btn_random.style.text_color = 'white'

btn_reset = widgets.Button(description="Reset", layout=widgets.Layout(flex='1'))
btn_reset.add_class('btn-action')
btn_load = widgets.Button(description="Load Example", layout=widgets.Layout(flex='1'))
btn_load.add_class('btn-action')

action_btns = widgets.HBox([btn_random, btn_reset, btn_load], layout=widgets.Layout(gap='10px'))

initial_state_card = widgets.VBox([
    widgets.HTML('<div class="card-header"><span>1. Initial State</span></div>'),
    input_grid, action_btns
], layout=widgets.Layout(margin='0 0 20px 0'))
initial_state_card.add_class('modern-card')

# 2. Khu vực AND-OR Configuration
btn_early = widgets.Button(description="AND-OR Search", layout=widgets.Layout(height='45px', flex='1'))
btn_early.add_class('btn-primary')

config_card = widgets.VBox([
    widgets.HTML('<div class="card-header"><span>AND-OR Search</span></div>'),
    widgets.VBox([btn_early], layout=widgets.Layout(width='100%', align_items='center'))
], layout=widgets.Layout(margin='0 0 20px 0')) 
config_card.add_class('modern-card')

# 3. Khu vực Visual Simulation 
step_label = widgets.HTML('<span style="background:#e1e8fd; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#004ac6;">Ready</span>')
sim_header = widgets.HBox([
    widgets.HTML('<span style="font-size: 18px; font-weight: 600; color: #141b2b;">3. Visual Simulation</span>'),
    step_label
], layout=widgets.Layout(justify_content='space-between', border_bottom='1px solid #c3c6d7', padding='0 0 15px 0', margin='0 0 20px 0', width='100%'))

anim_html = widgets.HTML(value="")
sim_card = widgets.VBox([sim_header, anim_html], layout=widgets.Layout(flex='1'))
sim_card.add_class('modern-card')

# Gom nhóm cột trái 
left_col = widgets.VBox([initial_state_card, sim_card], layout=widgets.Layout(flex='1.6', min_width='50%'))

# 4. Khu vực Thống kê (Stats Grid)
stat_steps = widgets.HTML()
stat_nodes = widgets.HTML()
stat_time = widgets.HTML()
stat_depth = widgets.HTML()

def update_stat(html_widget, value):
    html_widget.value = f'<div style="font-size: 28px; font-weight: 700; color: #141b2b;">{value}</div>'

update_stat(stat_steps, "-")
update_stat(stat_nodes, "-")
update_stat(stat_time, "-")
update_stat(stat_depth, "-")

def make_stat_box(title, html_widget):
    box = widgets.VBox([
        widgets.HTML(f'<span style="font-size: 11px; font-weight: 600; color: #54647a; text-transform: uppercase;">{title}</span>'),
        html_widget
    ], layout=widgets.Layout(width='100%'))
    box.add_class('stat-box')
    return box

stat_grid = widgets.GridBox([
    make_stat_box("Steps", stat_steps),
    make_stat_box("Nodes", stat_nodes),
    make_stat_box("Time", stat_time),
    make_stat_box("Max Depth", stat_depth)
], layout=widgets.Layout(grid_template_columns="1fr 1fr", gap="15px", margin="0 0 20px 0"))

# Khu vực Log kết quả chạy thuật toán
log_content = widgets.HTML(value='')
out_text = widgets.VBox([log_content], layout=widgets.Layout(flex='1', overflow='auto', padding='15px', max_height='350px'))
out_text.add_class('log-output')

log_card = widgets.VBox([
    widgets.HTML('<div style="display:flex; align-items:center; justify-content:space-between; border-bottom: 1px solid #c3c6d7; padding: 12px 20px; background-color: #e1e8fd; border-radius: 12px 12px 0 0;"><span style="font-size: 13px; font-weight: 700; color: #141b2b; text-transform: uppercase; letter-spacing: 0.5px;">Execution Log</span><div style="display:flex; gap: 5px;"><span style="color:#54647a; font-size:16px;">📋</span><span style="color:#54647a; font-size:16px;">⬇️</span></div></div>'),
    out_text
], layout=widgets.Layout(background_color='#f1f3ff', border='1px solid #c3c6d7', border_radius='12px', flex='1'))

# Xếp nhóm cột phải
right_col = widgets.VBox([
    stat_grid, 
    config_card, 
    log_card
], layout=widgets.Layout(flex='1', min_width='330px'))

# Tổng hợp ứng dụng vào container chính
main_app = widgets.VBox([
    header_html,
    widgets.HBox([left_col, right_col], layout=widgets.Layout(padding='20px', gap='20px'))
])
main_app.add_class('app-container')

# --- Logic Core ---
def get_successors(mt):
    pos = mt.index(0)
    r, c = pos // 3, pos % 3
    successors = []
    
    def swap(mt, i, j):
        new_mt = list(mt)
        new_mt[i], new_mt[j] = new_mt[j], new_mt[i]
        return new_mt
        
    if c > 0: successors.append(("Trái", swap(mt, pos, pos - 1)))
    if c < 2: successors.append(("Phải", swap(mt, pos, pos + 1)))
    if r > 0: successors.append(("Lên", swap(mt, pos, pos - 3)))
    if r < 2: successors.append(("Xuống", swap(mt, pos, pos + 3)))
    return successors

def and_or_graph_search(start_state, goal_state, limit):
    nodes_generated = 1
    
    def get_actions(state):
        pos = state.index(0)
        r, c = pos // 3, pos % 3
        actions = []
        if c > 0: actions.append("Trái")
        if c < 2: actions.append("Phải")
        if r > 0: actions.append("Lên")
        if r < 2: actions.append("Xuống")
        return actions

    def get_result(state, action):
        pos = state.index(0)
        new_mt = list(state)
        if action == "Trái":
            new_mt[pos], new_mt[pos - 1] = new_mt[pos - 1], new_mt[pos]
        elif action == "Phải":
            new_mt[pos], new_mt[pos + 1] = new_mt[pos + 1], new_mt[pos]
        elif action == "Lên":
            new_mt[pos], new_mt[pos - 3] = new_mt[pos - 3], new_mt[pos]
        elif action == "Xuống":
            new_mt[pos], new_mt[pos + 3] = new_mt[pos + 3], new_mt[pos]
        return [new_mt]

    def or_search(state, path):
        nonlocal nodes_generated
        if state == goal_state:
            return []
        if state in path:
            return "failure"
        if len(path) >= limit:
            return "failure"
            
        for action in get_actions(state):
            result_states = get_result(state, action)
            for r_state in result_states:
                nodes_generated += 1
            plan = and_search(result_states, path + [state])
            if plan != "failure":
                return [action, plan]
        return "failure"

    def and_search(states, path):
        plans = {}
        for s in states:
            plan_s = or_search(s, path)
            if plan_s == "failure":
                return "failure"
            plans[tuple(s)] = plan_s
        return plans

    plan = or_search(start_state, [])
    return plan, nodes_generated

def plan_to_path(plan):
    if plan == "failure":
        return None
    path = []
    current_plan = plan
    while current_plan:
        action = current_plan[0]
        plans = current_plan[1]
        if not plans:
            break
        child_state_tuple = list(plans.keys())[0]
        child_state = list(child_state_tuple)
        path.append((action, child_state))
        current_plan = plans[child_state_tuple]
    return path

def in_mt(mt):
    res = ""
    for i in range(3):
        row = ""
        for j in range(3):
            val = mt[i*3 + j]
            if val == 0: row += " [ ] "
            else: row += f"  {val}  "
        res += row + "\n"
    res += "-" * 20 + "\n"
    return res

def print_log_state(title, state, action=None):
    state_str = ""
    for i in range(0, 9, 3):
        row = state[i:i+3]
        state_str += "  " + "    ".join([str(x) if x != 0 else "[ ]" for x in row]) + "\n"
    
    if action:
        action_html = f'<p style="margin-bottom: 8px; font-weight: 600; color: #141b2b;">Bước {title}: Di chuyển ô trống sang <span style="color: #004ac6;">{action}</span></p>'
    else:
        action_html = f'<p style="margin-bottom: 8px; font-weight: 600; color: #141b2b;">{title}</p>'
        
    log_html = f'''
    <div style="margin-bottom: 15px;">
        {action_html}
        <div style="background-color: #ffffff; padding: 12px; border-radius: 8px; border: 1px solid rgba(195, 198, 215, 0.5); display: inline-block;">
            <pre style="margin: 0; font-family: 'JetBrains Mono', monospace; font-size: 13px; line-height: 1.4; color: #141b2b;">{state_str}</pre>
        </div>
    </div>
    <div style="border-top: 1px solid rgba(195, 198, 215, 0.5); margin-bottom: 15px; width: 100%;"></div>
    '''
    log_content.value += log_html

def render_board(state):
    html_content = '<div class="anim-board">'
    for val in state:
        if val == 0: html_content += '<div class="anim-tile-empty"></div>'
        else: html_content += f'<div class="anim-tile">{val}</div>'
    html_content += '</div>'
    anim_html.value = html_content

def animate_path(start_state, path):
    render_board(start_state)
    step_label.value = f'<span style="background:#e1e8fd; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#004ac6;">Step 0/{len(path)}</span>'
    time.sleep(1)
    for step, (action, state) in enumerate(path):
        render_board(state)
        step_label.value = f'<span style="background:#e1e8fd; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#004ac6;">Step {step + 1}/{len(path)}</span>'
        time.sleep(0.6)
    step_label.value = f'<span style="background:#d1f4e0; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#0d6e35;">Finished</span>'

def solve_and_animate():
    log_content.value = ''
    start_state = [box.value for box in input_boxes]
    goal_state = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    
    limit = 100  # Mặc định tối đa cho search thông thường
    log_title = "AND-OR SEARCH"
    
    render_board(start_state)
    log_content.value += f'<div style="color: #004ac6; font-weight: bold; margin-bottom: 10px; font-family: Inter;">ĐANG GIẢI BẰNG: {log_title}</div>'
    start_time = time.time()
    
    plan, nodes_generated = and_or_graph_search(start_state, goal_state, limit)
    path = plan_to_path(plan)
    
    end_time = time.time()
    elapsed_ms = int((end_time - start_time) * 1000)
    
    if path is not None:
        update_stat(stat_steps, str(len(path)))
        update_stat(stat_nodes, f"{nodes_generated:,}")
        update_stat(stat_time, f"{elapsed_ms}ms")
        update_stat(stat_depth, str(len(path)))
        
        print_log_state("Trạng thái bắt đầu:", start_state)
        for step, (action, state) in enumerate(path):
            print_log_state(str(step + 1), state, action)
            
        thread = threading.Thread(target=animate_path, args=(start_state, path))
        thread.start()
    else:
        log_content.value += '<div style="color: red; font-weight: bold; font-family: Inter;">Không tìm thấy giải pháp!</div>'
        update_stat(stat_steps, "-")
        update_stat(stat_nodes, f"{nodes_generated:,}")
        update_stat(stat_time, f"{elapsed_ms}ms")
        update_stat(stat_depth, "-")

btn_early.on_click(lambda b: solve_and_animate())

def randomize_board(b):
    nums = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    random.shuffle(nums)
    for i, box in enumerate(input_boxes): box.value = nums[i]
    render_board(nums)
btn_random.on_click(randomize_board)

def reset_board(b):
    nums = [0]*9
    for i, box in enumerate(input_boxes): box.value = nums[i]
    render_board(nums)
btn_reset.on_click(reset_board)

def load_example(b):
    nums = [1, 2, 3, 4, 0, 5, 7, 8, 6]
    for i, box in enumerate(input_boxes): box.value = nums[i]
    render_board(nums)
btn_load.on_click(load_example)

# Khởi chạy ban đầu
render_board([1, 2, 3, 4, 0, 5, 7, 8, 6])
display(main_app)